# 🎙️ Real-Time Voice AI — Speech-to-Speech Conversation
> **Runtime**: T4 GPU | Record → Stop → AI responds as speech → Repeat

**Run all cells top to bottom. Launch happens in the last cell.**

In [1]:
# ─── Cell 1: System packages ───────────────────────────────────────────────
!apt-get update -qq
!apt-get install -y -qq ffmpeg espeak-ng libsndfile1
print('✅ System packages installed')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.1-6build2_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.1-6build2) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-11build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-11build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../espeak-ng-data_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../libespeak-ng1_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previ

In [2]:
# ─── Cell 2: Python packages ───────────────────────────────────────────────
!pip install -q faster-whisper "kokoro>=0.9.4" soundfile gradio transformers accelerate torch
print('✅ Python packages installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 55.2 MB/s eta 0:00:00
   ━━━━━━

In [4]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.7 MB/s eta 0:00:00


In [ ]:
from groq import Groq

GROQ_API_KEY = ''   # ← paste your key from console.groq.com
MODEL        = 'llama-3.1-8b-instant'

client = Groq(api_key=GROQ_API_KEY)

print('🔍 Testing Groq API connection...')
try:
    _test = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'Reply with exactly the word: OK'}],
        max_tokens=5
    )
    print(f'✅ Groq API working! Reply: {_test.choices[0].message.content.strip()}')
    print(f'✅ Model "{MODEL}" is ready!')
except Exception as e:
    print(f'❌ Groq API FAILED: {e}')
    print('👉 Get a free key at: https://console.groq.com')
    raise

🔍 Testing Groq API connection...
✅ Groq API working! Reply: OK
✅ Model "llama-3.1-8b-instant" is ready!


In [6]:
from faster_whisper import WhisperModel
from kokoro import KPipeline
import numpy as np
import soundfile as sf
import torch

print(f'🖥️  GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   → {torch.cuda.get_device_name(0)}')
else:
    print('   ⚠️  No GPU — go to Runtime > Change runtime type > T4 GPU')

print('\n⏳ Loading Whisper small...')
stt = WhisperModel(
    'small',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    compute_type='float16' if torch.cuda.is_available() else 'int8'
)
print('✅ Whisper ready!')

print('\n⏳ Loading Kokoro TTS...')
tts = KPipeline(lang_code='a')
print('⏳ Pre-warming TTS...')
_ = list(tts('Hello!', voice='af_sky', speed=1.0))
print('✅ Kokoro TTS ready!')

print('\n🎉 All models loaded!')

🖥️  GPU available: True
   → Tesla T4

⏳ Loading Whisper small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Whisper ready!

⏳ Loading Kokoro TTS...


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

⏳ Pre-warming TTS...


voices/af_sky.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

✅ Kokoro TTS ready!

🎉 All models loaded!


In [7]:
import time
import numpy as np
import soundfile as sf

EMOTION_CONFIG = {
    'happy':   {'voice': 'af_heart', 'speed': 1.15},
    'sad':     {'voice': 'af_sky',   'speed': 0.82},
    'angry':   {'voice': 'am_adam',  'speed': 1.05},
    'excited': {'voice': 'af_heart', 'speed': 1.25},
    'calm':    {'voice': 'af_sky',   'speed': 0.95},
    'neutral': {'voice': 'af_sky',   'speed': 1.00},
}

EMOTION_KEYWORDS = {
    'angry':   ['angry', 'furious', 'frustrated', 'unacceptable', 'terrible', 'worst'],
    'sad':     ['sad', 'upset', 'disappointed', 'depressed', 'unhappy', 'horrible'],
    'happy':   ['happy', 'amazing', 'love', 'wonderful', 'fantastic', 'thank you'],
    'excited': ['excited', 'awesome', 'thrilled', 'incredible', "can't wait"],
}

EMOTION_MAP = {
    'angry': 'calm', 'sad': 'sad', 'happy': 'happy',
    'excited': 'excited', 'neutral': 'neutral'
}


def detect_emotion(text):
    t = text.lower()
    for emotion, kws in EMOTION_KEYWORDS.items():
        if any(k in t for k in kws):
            mapped = EMOTION_MAP.get(emotion, 'neutral')
            print(f'   🎭 Detected: {emotion} → AI voice: {mapped}')
            return mapped
    return 'neutral'


def transcribe(audio_array, sr):
    """numpy audio → text via Whisper."""
    if sr != 16000:
        try:
            import librosa
            audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
        except ImportError:
            ratio = 16000 / sr
            n = int(len(audio_array) * ratio)
            audio_array = np.interp(
                np.linspace(0, len(audio_array) - 1, n),
                np.arange(len(audio_array)),
                audio_array
            )
        sr = 16000

    sf.write('/tmp/user_audio.wav', audio_array, sr)

    segments, info = stt.transcribe(
        '/tmp/user_audio.wav',
        beam_size=5,
        language='en',                        # ← FIXED: force English only
        vad_filter=True,
        vad_parameters=dict(
            min_silence_duration_ms=300,
            speech_pad_ms=400,
            threshold=0.5                     # ← stricter VAD — ignores noise
        ),
        condition_on_previous_text=False,
        temperature=0.0,
        no_speech_threshold=0.6,              # ← FIXED: reject low-confidence audio
        compression_ratio_threshold=2.0,      # ← FIXED: detect hallucination loops
        log_prob_threshold=-0.8               # ← FIXED: reject uncertain transcriptions
    )

    text = ' '.join(s.text for s in segments).strip()

    # ── Hallucination detection ────────────────────────────────────────────
    # If any single word repeats more than 4 times → it's a hallucination
    words = text.split()
    if len(words) > 4:
        from collections import Counter
        word_counts = Counter(words)
        most_common_word, most_common_count = word_counts.most_common(1)[0]
        if most_common_count > 4:
            print(f'   ⚠️  Hallucination detected: "{most_common_word}" repeated {most_common_count}x — ignoring')
            return ''

    print(f'   🎤 [{info.language}] "{text}"')
    return text


def ask_ai(user_text, history):
    """Send text + history to Groq → AI reply."""

    if not user_text or len(user_text.strip()) < 3:
        print('   ⚠️  Transcription too short, skipping')
        return "I didn't catch that — could you speak a little louder?"

    print(f'   📤 Sending to Groq: "{user_text}"')

    messages = [
        {
            'role': 'system',
            'content': (
                'You are a friendly, empathetic AI voice assistant having a natural '
                'phone conversation. Keep every reply to 1-2 short conversational '
                'sentences. Do NOT use bullet points, lists, or markdown. Speak naturally.'
            )
        }
    ]

    for turn in history[-6:]:
        messages.append({'role': 'user',      'content': turn[0]})
        messages.append({'role': 'assistant', 'content': turn[1]})

    messages.append({'role': 'user', 'content': user_text})

    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            max_tokens=80,
            temperature=0.75,
            top_p=0.9,
        )
        reply = resp.choices[0].message.content.strip()
        print(f'   🤖 AI reply: "{reply}"')
        if not reply:
            return "I'm not sure I caught that — could you say it again?"
        return reply

    except Exception as ex:
        print(f'   ❌ Groq error [{type(ex).__name__}]: {ex}')
        return "Sorry, I had a brief hiccup. Please go ahead."


def speak(text, emotion='neutral'):
    """text → (sample_rate, audio_array) via Kokoro TTS."""
    cfg = EMOTION_CONFIG.get(emotion, EMOTION_CONFIG['neutral'])
    try:
        chunks = [audio for _, _, audio in tts(text, voice=cfg['voice'], speed=cfg['speed'])]
        if not chunks:
            raise ValueError('TTS returned empty audio')
        return (24000, np.concatenate(chunks).astype(np.float32))
    except Exception as ex:
        print(f'   ❌ TTS error: {ex}')
        return (24000, np.zeros(12000, dtype=np.float32))


print('✅ All helper functions ready!')

✅ All helper functions ready!


In [8]:
# ─── Cell 6: Non-streaming pipeline (record → stop → send) ────────────────
import gradio as gr
import numpy as np
import time


def make_state():
    return {'history': []}


def process_audio(audio, state):
    """Called when user clicks Send to AI button."""
    if state is None:
        state = make_state()

    if audio is None:
        return (state, '', 'Please record something first.', 'neutral',
                None, '❌ No audio recorded — press Record first')

    sr, data = audio

    # Normalize
    if data.dtype != np.float32:
        data = data.astype(np.float32) / 32768.0
    if data.ndim == 2:
        data = data.mean(axis=1)

    # Check if audio is too short or silent
    duration = len(data) / sr
    energy   = float(np.abs(data).mean())
    print(f'   📊 Audio: {duration:.1f}s, energy: {energy:.4f}')

    if duration < 0.5:
        return (state, '', 'Audio too short — please speak for at least 1 second.',
                'neutral', None, '⚠️ Too short — try again')

    if energy < 0.001:
        return (state, '', 'Audio too quiet — please speak louder.',
                'neutral', None, '⚠️ Too quiet — speak louder and try again')

    history = state['history']
    t0 = time.time()

    # 1. Speech → Text
    try:
        t = time.time()
        user_text = transcribe(data, sr)
        stt_ms = (time.time() - t) * 1000
        print(f'   ✅ Transcribed in {stt_ms:.0f}ms: "{user_text}"')
    except Exception as ex:
        print(f'   ❌ STT error: {ex}')
        return (state, '', 'Speech recognition failed — try again.',
                'neutral', None, '❌ STT error')

    if not user_text or len(user_text.strip()) < 2:
        return (state, '[nothing detected]',
                "I couldn't hear you clearly — please try again.",
                'neutral', None, '⚠️ Nothing heard — speak clearly')

    # 2. Text → AI reply
    t = time.time()
    ai_text = ask_ai(user_text, history)
    llm_ms  = (time.time() - t) * 1000

    # 3. Detect emotion
    emotion = detect_emotion(user_text)

    # 4. Text → Speech
    t = time.time()
    audio_out = speak(ai_text, emotion)
    tts_ms    = (time.time() - t) * 1000

    total_ms = (time.time() - t0) * 1000
    history  = (history + [(user_text, ai_text)])[-10:]
    state    = {'history': history}

    timing = (
        f'🎤 STT:  {stt_ms:.0f} ms\n'
        f'🤖 LLM:  {llm_ms:.0f} ms\n'
        f'🔊 TTS:  {tts_ms:.0f} ms\n'
        f'──────────────\n'
        f'⚡ TOTAL: {total_ms:.0f} ms ({total_ms/1000:.1f}s)'
    )

    print(f'👤 User: {user_text}')
    print(f'🤖 AI:   {ai_text} [{emotion}]')
    print(timing)

    return (state, user_text, ai_text, emotion, audio_out,
            '✅ AI spoke! Record again to continue...')


print('✅ Handler ready!')

✅ Handler ready!


In [9]:
# ─── Cell 7: Gradio UI ────────────────────────────────────────────────────
import gradio as gr

CSS = """
.container { max-width: 820px; margin: auto; }
.status-box textarea { font-size: 15px !important; font-weight: 600; }
.send-btn { background: #6d28d9 !important; font-size: 16px !important; }
"""

with gr.Blocks(title='Voice AI', css=CSS, theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎙️ Real-Time Voice AI
    **Steps:** 1️⃣ Press **Record** → 2️⃣ Speak → 3️⃣ Press **Stop** → 4️⃣ Click ▶ **Send to AI** → 5️⃣ Listen → Repeat
    > Whisper (STT) + Llama 3.1-8b via Groq (LLM) + Kokoro (TTS) on T4 GPU
    """)

    state = gr.State({'history': []})

    status_box = gr.Textbox(
        value='🎤 Ready — press Record and speak',
        label='📡 Status',
        interactive=False,
        max_lines=1,
        elem_classes='status-box'
    )

    # Microphone — non-streaming, user controls recording
    mic = gr.Audio(
        sources=['microphone'],
        streaming=False,       # ← KEY CHANGE: non-streaming
        type='numpy',
        label='🎤 Microphone  (Record → speak → Stop)'
    )

    # Send button — user clicks this after recording
    send_btn = gr.Button('▶  Send to AI', variant='primary', elem_classes='send-btn')

    with gr.Row():
        user_box = gr.Textbox(
            label='🧑 You said',
            max_lines=4,
            interactive=False,
            placeholder='Your speech will appear here...'
        )
        ai_box = gr.Textbox(
            label='🤖 AI Response',
            max_lines=4,
            interactive=False,
            placeholder='AI reply will appear here...'
        )

    emotion_box = gr.Textbox(
        label='🎭 Detected Emotion',
        max_lines=1,
        interactive=False
    )

    ai_audio = gr.Audio(
        label='🔊 AI Voice — plays automatically',
        autoplay=True,
        interactive=False
    )

    # Wire send button to handler
    send_btn.click(
        fn=process_audio,
        inputs=[mic, state],
        outputs=[state, user_box, ai_box, emotion_box, ai_audio, status_box]
    )

    gr.Markdown("""
    ---
    **💡 Tips:**
    - Press **Record**, speak clearly, press **Stop**, then click **Send to AI**
    - Wait for **"✅ AI spoke!"** before recording again
    - AI remembers last 10 turns of conversation
    """)

demo.launch(share=True, debug=True, quiet=False)

/tmp/ipykernel_509/112505963.py:10: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Voice AI', css=CSS, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_509/112505963.py:10: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title='Voice AI', css=CSS, theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9401cfaec0fb9bb73c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9401cfaec0fb9bb73c.gradio.live
